# LeetCode 1561 — Maximum Number of Coins You Can Get bằng Python

Link đề: https://leetcode.com/problems/maximum-number-of-coins-you-can-get/ · Độ khó: **Medium** · Chủ đề: Array, Math, Greedy, Sorting, Game Theory

Skeleton LeetCode:

```python
class Solution:
    def maxCoins(self, piles: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đọc hiểu luật chơi và nhận ra **bạn là người chọn bộ ba** — đây là bài tối ưu hoá phân hoạch, không phải trò chơi đối kháng thật sự.
2. Viết baseline **vét cạn mọi cách chia bộ ba** (đệ quy) để có chuẩn đối chiếu.
3. Nắm lời giải tham lam: **sắp xếp**, cho Bob `n` cọc nhỏ nhất, còn lại bạn lấy xen kẽ từ trên xuống — tức `sum(sorted(piles)[n::2])` với `n = len(piles) // 3`.
4. Chứng minh tham lam tối ưu bằng lập luận **đếm / cận trên**: cọc lớn thứ `k` của bạn không vượt quá cọc lớn thứ `2k` của cả mảng.
5. Biết thêm cách hai con trỏ và **sắp xếp đếm** `O(n + V)`; tránh các lỗi lệch chỉ số khi cắt mảng.

## 2. Đề bài & ví dụ

Có `3n` cọc xu với số xu khác nhau. Bạn, Alice và Bob lấy xu như sau:

- Mỗi bước, **bạn** chọn 3 cọc bất kỳ (không cần liền kề).
- Alice lấy cọc **nhiều xu nhất** trong 3 cọc.
- Bạn lấy cọc **nhiều thứ hai**.
- Bob lấy cọc còn lại.
- Lặp lại cho tới khi hết cọc.

Cho mảng `piles` (`piles[i]` = số xu của cọc `i`), trả về **số xu lớn nhất bạn có thể có**.

**Ví dụ 1:** `piles = [2,4,1,2,7,8]` → `9`. Chọn `(2,7,8)`: Alice 8, bạn 7, Bob 2. Chọn `(1,2,4)`: Alice 4, bạn 2, Bob 1. Tổng `7 + 2 = 9`. Nếu chọn `(1,2,8)`, `(2,4,7)` bạn chỉ được `2 + 4 = 6`.

**Ví dụ 2:** `piles = [2,4,5]` → `4`.

**Ví dụ 3:** `piles = [9,8,7,6,5,1,2,3,4]` → `18`.

**Ràng buộc:** `3 ≤ piles.length ≤ 10⁵`, `piles.length % 3 == 0`, `1 ≤ piles[i] ≤ 10⁴`.

In [1]:
import random
import time
from itertools import combinations
from typing import List

examples = [([2, 4, 1, 2, 7, 8], 9), ([2, 4, 5], 4), ([9, 8, 7, 6, 5, 1, 2, 3, 4], 18)]
for piles, want in examples:
    print(f"piles = {piles} -> mong đợi {want}")

piles = [2, 4, 1, 2, 7, 8] -> mong đợi 9
piles = [2, 4, 5] -> mong đợi 4
piles = [9, 8, 7, 6, 5, 1, 2, 3, 4] -> mong đợi 18


## 3. Kiến thức nền

### 3.1 Sắp xếp và cắt lát có bước nhảy

`a[start:stop:step]` lấy các phần tử từ `start`, mỗi lần nhảy `step`. `a[2::2]` = các phần tử ở chỉ số `2, 4, 6, ...`; `a[-2::-2]` = từ phần tử kế cuối đi lùi 2.

In [2]:
a = sorted([9, 8, 7, 6, 5, 1, 2, 3, 4])
print("sorted   :", a)
print("a[3::2]  :", a[3::2])
print("a[-2::-2]:", a[-2::-2])

sorted   : [1, 2, 3, 4, 5, 6, 7, 8, 9]
a[3::2]  : [4, 6, 8]
a[-2::-2]: [8, 6, 4, 2]


### 3.2 Phần tử "lớn thứ hai" của bộ ba

`sorted(t)[1]` là phần tử giữa (lớn thứ hai) của bộ ba `t`. Đó chính là phần của bạn trong mỗi bước.

In [3]:
for t in [(2, 7, 8), (1, 2, 4), (1, 2, 8), (2, 4, 7)]:
    print(t, "-> Alice", max(t), "| bạn", sorted(t)[1], "| Bob", min(t))

(2, 7, 8) -> Alice 8 | bạn 7 | Bob 2
(1, 2, 4) -> Alice 4 | bạn 2 | Bob 1
(1, 2, 8) -> Alice 8 | bạn 2 | Bob 1
(2, 4, 7) -> Alice 7 | bạn 4 | Bob 2


### 3.3 Hai con trỏ trên mảng đã sắp

Con trỏ `lo` từ đầu (cọc nhỏ — cho Bob), con trỏ `hi` từ cuối (cọc lớn — Alice và bạn). Mỗi vòng `hi` lùi 2, `lo` tiến 1.

In [4]:
a = [1, 2, 2, 4, 7, 8]
lo, hi = 0, len(a) - 1
while lo < hi:
    print(f"Bob a[{lo}]={a[lo]}, bạn a[{hi-1}]={a[hi-1]}, Alice a[{hi}]={a[hi]}")
    lo += 1
    hi -= 2

Bob a[0]=1, bạn a[4]=7, Alice a[5]=8
Bob a[1]=2, bạn a[2]=2, Alice a[3]=4


### 3.4 Sắp xếp đếm (counting sort)

Khi giá trị nhỏ (`≤ 10⁴`), đếm tần suất bằng mảng `cnt[v]` rồi duyệt `v` từ lớn xuống — `O(n + V)` thay vì `O(n log n)`.

In [5]:
piles = [2, 4, 1, 2, 7, 8]
cnt = [0] * (max(piles) + 1)
for x in piles:
    cnt[x] += 1
print([(v, cnt[v]) for v in range(len(cnt)) if cnt[v]])

[(1, 1), (2, 2), (4, 1), (7, 1), (8, 1)]


### 3.5 Đệ quy liệt kê phân hoạch thành bộ ba

Để không đếm trùng, luôn đặt **phần tử đầu tiên còn lại** vào bộ ba hiện tại, rồi chọn 2 phần tử nữa trong phần còn lại. Số cách chia `3n` phần tử thành `n` bộ ba (không thứ tự) là `(3n)! / (3!ⁿ · n!)`.

In [6]:
from math import factorial
for n in [1, 2, 3, 4]:
    print(f"3n = {3*n}: số cách chia bộ ba = {factorial(3*n) // (6**n * factorial(n))}")

3n = 3: số cách chia bộ ba = 1
3n = 6: số cách chia bộ ba = 10
3n = 9: số cách chia bộ ba = 280
3n = 12: số cách chia bộ ba = 15400


## 4. Ý tưởng nền tảng / trực giác

**Ai lấy gì không phải do may rủi**: bạn chọn bộ ba, nên bạn quyết định *Bob nhận cọc nào*. Bob luôn nhận cọc nhỏ nhất của bộ ba — vậy hãy "đẩy" cho Bob những cọc **tệ nhất** cả mảng.

Tính tay ví dụ 3: `piles` đã sắp = `[1,2,3,4,5,6,7,8,9]`, `n = 3`.

- Bob nên nhận 3 cọc nhỏ nhất: `1, 2, 3`.
- Còn lại `4,5,6,7,8,9` chia cho Alice và bạn. Mỗi bộ ba Alice luôn lấy phần lớn hơn bạn, nên ghép theo cặp từ trên xuống: `(9, 8)`, `(7, 6)`, `(5, 4)` — Alice lấy `9, 7, 5`, bạn lấy `8, 6, 4`.
- Bộ ba thực tế: `(1, 8, 9)`, `(2, 6, 7)`, `(3, 4, 5)`. Bạn được `8 + 6 + 4 = 18`.

Trong mảng đã sắp, phần của bạn chính là `a[3], a[5], a[7]` = `a[n::2]`.

In [7]:
a = sorted([9, 8, 7, 6, 5, 1, 2, 3, 4])
n = len(a) // 3
print("Bob nhận:", a[:n])
print("Alice nhận:", a[n + 1::2])
print("Bạn nhận:", a[n::2], "-> tổng", sum(a[n::2]))

Bob nhận: [1, 2, 3]
Alice nhận: [5, 7, 9]
Bạn nhận: [4, 6, 8] -> tổng 18


## 5. Cách 1 — Brute Force / baseline: thử mọi cách chia bộ ba

**Ý tưởng.** Đệ quy: lấy phần tử đầu tiên còn lại `x`, thử mọi cặp `(y, z)` trong phần còn lại để tạo bộ ba `(x, y, z)`; bạn được phần tử giữa của bộ ba; đệ quy tiếp với phần còn lại; lấy max. Thứ tự các bước không ảnh hưởng tổng nên chỉ cần quan tâm *cách chia*.

**Độ phức tạp:** `(3n)! / (6ⁿ n!)` cách chia — `n = 4` (12 cọc) đã là 15 400 cách, `n = 6` là ~19 triệu. Chỉ dùng cho `len(piles) ≤ 12`.

In [8]:
def brute_partitions(piles):
    def solve(rest):
        if not rest:
            return 0
        x, others = rest[0], rest[1:]
        best = 0
        for i, j in combinations(range(len(others)), 2):
            trio = sorted((x, others[i], others[j]))
            remain = [others[k] for k in range(len(others)) if k != i and k != j]
            best = max(best, trio[1] + solve(remain))
        return best
    return solve(list(piles))

for piles, want in examples:
    print(brute_partitions(piles), "mong đợi", want)

9 mong đợi 9
4 mong đợi 4
18 mong đợi 18


In [9]:
# Nút thắt: số cách chia tăng siêu nhanh
rng = random.Random(0)
for size in [6, 9, 12]:
    piles = [rng.randint(1, 50) for _ in range(size)]
    t0 = time.perf_counter()
    r = brute_partitions(piles)
    print(f"{size} cọc: {r} ({time.perf_counter() - t0:.3f}s)")
for size in [15, 30, 60]:
    k = size // 3
    ways = factorial(size) // (6**k * factorial(k))
    print(f"{size} cọc: số cách chia có {len(str(ways))} chữ số" + (f" ({ways:,})" if size == 15 else ""))

6 cọc: 58 (0.000s)
9 cọc: 87 (0.000s)
12 cọc: 122 (0.019s)
15 cọc: số cách chia có 7 chữ số (1,401,400)
30 cọc: số cách chia có 19 chữ số
60 cọc: số cách chia có 48 chữ số


**Nút thắt:** thử mọi phân hoạch là bùng nổ tổ hợp. Cần biết **cấu trúc** của phân hoạch tối ưu để xây thẳng nó.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — Bob là "thùng rác".** Mỗi bộ ba Bob lấy cọc nhỏ nhất. Bạn muốn Bob lấy những cọc vô dụng nhất ⇒ `n` cọc nhỏ nhất cả mảng thuộc về Bob.

**Insight 2 — với Alice thì ghép cặp từ trên xuống.** Trong `2n` cọc lớn còn lại, mỗi bộ ba có một cọc của Alice và một cọc của bạn với `Alice ≥ bạn`. Cọc lớn nhất chắc chắn thuộc Alice (không ai trong bộ ba lớn hơn nó). Cọc lớn thứ hai thì tốt nhất là của bạn — ghép nó cùng bộ ba với cọc lớn nhất. Lặp lại: `(a₁, a₂), (a₃, a₄), ...` theo thứ tự giảm dần, bạn lấy các vị trí chẵn `a₂, a₄, ..., a₂ₙ`.

**Quy tắc:** *nếu* sắp `piles` tăng dần *thì* đáp án = `sum(piles[n::2])` với `n = len(piles) // 3` (bỏ `n` cọc đầu cho Bob, rồi lấy xen kẽ bắt đầu từ cọc nhỏ hơn trong mỗi cặp).

**Cận trên (giải thích vì sao không làm tốt hơn được):** gọi các cọc của bạn theo thứ tự giảm `y₁ ≥ y₂ ≥ ... ≥ yₙ`. Với mỗi `k`, có `k` cọc của bạn `≥ y_k` và mỗi cọc đó đi kèm một cọc của Alice còn lớn hơn ⇒ ít nhất `2k` cọc `≥ y_k` ⇒ `y_k ≤ a_{2k}` (cọc lớn thứ `2k`). Cộng lại: tổng của bạn `≤ a₂ + a₄ + ... + a₂ₙ` — đúng bằng giá trị tham lam đạt được.

In [10]:
a = sorted([2, 4, 1, 2, 7, 8], reverse=True)
n = len(a) // 3
print("giảm dần:", a)
print("cận trên a2 + a4 + ... =", [a[2 * k - 1] for k in range(1, n + 1)], "->", sum(a[2 * k - 1] for k in range(1, n + 1)))

giảm dần: [8, 7, 4, 2, 2, 1]
cận trên a2 + a4 + ... = [7, 2] -> 9


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Sắp xếp + cắt lát

Từng đoạn:
1. `piles.sort()` — tăng dần.
2. `n = len(piles) // 3` — số bước, cũng là số cọc của Bob.
3. `sum(piles[n::2])` — bỏ `n` cọc nhỏ nhất, lấy xen kẽ.

Tại sao bắt đầu ở `n` mà không phải `n + 1`? Phần `piles[n:]` có `2n` phần tử, xếp tăng: `(b₁, A₁), (b₂, A₂), ...` theo cặp (bạn, Alice) từ dưới lên — phần tử đầu mỗi cặp (chỉ số `n, n+2, ...`) là của bạn.

In [11]:
def sort_slice(piles):
    a = sorted(piles)
    n = len(a) // 3
    return sum(a[n::2])

for piles, want in examples:
    print(sort_slice(piles), "mong đợi", want)

9 mong đợi 9
4 mong đợi 4
18 mong đợi 18


Code nộp LeetCode cho cách 2:

In [12]:
class SolutionSlice:
    def maxCoins(self, piles: List[int]) -> int:
        piles.sort()
        return sum(piles[len(piles) // 3 :: 2])

print([SolutionSlice().maxCoins(list(p)) for p, _ in examples])

[9, 4, 18]


**Độ phức tạp:** Time `O(n log n)` (sắp xếp), Space `O(1)` thêm nếu sắp tại chỗ (lát cắt tạo list tạm `O(n)`; có thể tránh bằng vòng `range`).

### 7.2 Cách 3 — Hai con trỏ mô phỏng từng bước

Từng đoạn:
1. Sắp tăng dần; `lo = 0` (Bob), `hi = len − 1` (Alice).
2. Lặp `n` lần: bạn lấy `piles[hi − 1]`; `hi -= 2`; `lo += 1`.

Mô phỏng sát luật chơi, dễ giải thích khi phỏng vấn.

In [13]:
class SolutionTwoPointers:
    def maxCoins(self, piles: List[int]) -> int:
        piles.sort()
        lo, hi = 0, len(piles) - 1
        total = 0
        while lo < hi:
            total += piles[hi - 1]      # bạn lấy cọc lớn thứ hai của bộ (piles[lo], piles[hi-1], piles[hi])
            hi -= 2
            lo += 1
        return total

print([SolutionTwoPointers().maxCoins(list(p)) for p, _ in examples])

[9, 4, 18]


**Độ phức tạp:** Time `O(n log n)`, Space `O(1)`.

### 7.3 Cách 4 — Sắp xếp đếm `O(n + V)`

Vì `piles[i] ≤ 10⁴`, đếm tần suất rồi duyệt giá trị từ **lớn xuống**, phát lần lượt "Alice, bạn, Alice, bạn, ..." cho `2n` cọc đầu tiên gặp được. Bạn nhận các cọc ở lượt chẵn (thứ 2, 4, ...).

In [14]:
class SolutionCounting:
    def maxCoins(self, piles: List[int]) -> int:
        n = len(piles) // 3
        cnt = [0] * (max(piles) + 1)
        for x in piles:
            cnt[x] += 1
        total, taken = 0, 0                 # taken = số cọc đã phát cho Alice/bạn
        for v in range(len(cnt) - 1, 0, -1):
            c = cnt[v]
            while c and taken < 2 * n:
                if taken % 2 == 1:          # lượt thứ 2, 4, ... là của bạn
                    total += v
                taken += 1
                c -= 1
            if taken == 2 * n:
                break
        return total

print([SolutionCounting().maxCoins(list(p)) for p, _ in examples])

[9, 4, 18]


**Độ phức tạp:** Time `O(n + V)` với `V = max(piles) ≤ 10⁴`, Space `O(V)`. Vòng `while` tổng cộng chỉ chạy `2n` lần.

## 8. TRACE từng bước

Trace hai con trỏ trên ví dụ 1: sắp xếp `[1, 2, 2, 4, 7, 8]`, `n = 2`.

| Vòng | lo | hi | Bộ ba (Bob, bạn, Alice) | Bạn nhận | Tổng |
|------|----|----|-------------------------|----------|------|
| 1 | 0 | 5 | (1, 7, 8) | 7 | 7 |
| 2 | 1 | 3 | (2, 2, 4) | 2 | 9 |
| — | 2 | 1 | dừng (`lo ≥ hi`) | | **9** |

In [15]:
def trace_two_pointers(piles):
    a = sorted(piles)
    print("sắp xếp:", a)
    lo, hi = 0, len(a) - 1
    total, rnd = 0, 0
    while lo < hi:
        rnd += 1
        total += a[hi - 1]
        print(f"  vòng {rnd}: lo={lo}, hi={hi} -> bộ ba (Bob {a[lo]}, bạn {a[hi-1]}, Alice {a[hi]}), tổng = {total}")
        hi -= 2
        lo += 1
    print(f"  dừng: lo={lo}, hi={hi}, đáp án {total}")
    return total

trace_two_pointers([2, 4, 1, 2, 7, 8])

sắp xếp: [1, 2, 2, 4, 7, 8]
  vòng 1: lo=0, hi=5 -> bộ ba (Bob 1, bạn 7, Alice 8), tổng = 7
  vòng 2: lo=1, hi=3 -> bộ ba (Bob 2, bạn 2, Alice 4), tổng = 9
  dừng: lo=2, hi=1, đáp án 9


9

In [16]:
trace_two_pointers([9, 8, 7, 6, 5, 1, 2, 3, 4])

sắp xếp: [1, 2, 3, 4, 5, 6, 7, 8, 9]
  vòng 1: lo=0, hi=8 -> bộ ba (Bob 1, bạn 8, Alice 9), tổng = 8
  vòng 2: lo=1, hi=6 -> bộ ba (Bob 2, bạn 6, Alice 7), tổng = 14
  vòng 3: lo=2, hi=4 -> bộ ba (Bob 3, bạn 4, Alice 5), tổng = 18
  dừng: lo=3, hi=2, đáp án 18


18

Trace cách đếm — in từng giá trị được phát và người nhận:

In [17]:
def trace_counting(piles):
    n = len(piles) // 3
    cnt = [0] * (max(piles) + 1)
    for x in piles:
        cnt[x] += 1
    taken, total = 0, 0
    for v in range(len(cnt) - 1, 0, -1):
        for _ in range(cnt[v]):
            if taken == 2 * n:
                break
            who = "bạn" if taken % 2 else "Alice"
            if who == "bạn":
                total += v
            print(f"  cọc {v:>2} -> {who:5s} | tổng của bạn = {total}")
            taken += 1
    print("  phần còn lại cho Bob; đáp án", total)
    return total

trace_counting([2, 4, 1, 2, 7, 8])

  cọc  8 -> Alice | tổng của bạn = 0
  cọc  7 -> bạn   | tổng của bạn = 7
  cọc  4 -> Alice | tổng của bạn = 7
  cọc  2 -> bạn   | tổng của bạn = 9
  phần còn lại cho Bob; đáp án 9


9

## 9. Code nộp LeetCode

In [18]:
class Solution:
    def maxCoins(self, piles: List[int]) -> int:
        piles.sort()
        n = len(piles) // 3
        # Bob nhận n cọc nhỏ nhất; trong 2n cọc còn lại, bạn lấy xen kẽ (cọc nhỏ hơn mỗi cặp)
        return sum(piles[n::2])

for piles, want in examples:
    got = Solution().maxCoins(list(piles))
    print(got, want)
    assert got == want

9 9
4 4
18 18


## 10. Vì sao thuật toán đúng

Sắp `piles` giảm dần: `a₁ ≥ a₂ ≥ ... ≥ a₃ₙ`.

**Cận trên.** Xét một cách chia bất kỳ; các cọc của bạn theo thứ tự giảm là `y₁ ≥ ... ≥ yₙ`. Cố định `k`. Các cọc `y₁, ..., y_k` mỗi cái nằm trong một bộ ba khác nhau, và trong mỗi bộ ba đó Alice có cọc `≥` cọc của bạn `≥ y_k`. Vậy có ít nhất `2k` cọc (k của bạn + k của Alice) có giá trị `≥ y_k` ⇒ cọc lớn thứ `2k` của mảng `a_{2k} ≥ y_k`. Cộng theo `k`: `Σ y_k ≤ Σ a_{2k}`.

**Đạt được.** Chia bộ ba `(a_{2k−1}, a_{2k}, a_{2n+k})` với `k = 1..n`. Trong mỗi bộ, `a_{2k−1} ≥ a_{2k} ≥ a_{2n+k}` (vì `2n + k > 2k`) ⇒ Alice lấy `a_{2k−1}`, bạn lấy `a_{2k}`, Bob lấy `a_{2n+k}`. Tổng của bạn `= Σ a_{2k}` = cận trên.

Trong mảng tăng dần, `a_{2k}` (giảm dần) tương ứng các chỉ số `n, n+2, ..., 3n−2` ⇒ `sum(piles[n::2])`.

In [19]:
# thực nghiệm: cận trên Σ a_{2k} đúng bằng brute force trên mảng nhỏ
rng = random.Random(10)
for _ in range(300):
    size = rng.choice([3, 6, 9])
    piles = [rng.randint(1, 20) for _ in range(size)]
    a = sorted(piles, reverse=True)
    bound = sum(a[2 * k - 1] for k in range(1, size // 3 + 1))
    assert bound == brute_partitions(piles)
print("cận trên == brute force: OK")

cận trên == brute force: OK


## 11. Phân tích Time / Space

Gọi `N = len(piles) = 3n`, `V = max(piles) ≤ 10⁴`.

| Cách | Time | Space |
|------|------|-------|
| Vét cạn phân hoạch | `O(N!/(6ⁿ n!) · N)` | `O(N)` |
| Sort + cắt lát | `O(N log N)` | `O(N)` (lát cắt) / `O(1)` (vòng range) |
| Sort + hai con trỏ | `O(N log N)` | `O(1)` |
| Đếm | `O(N + V)` | `O(V)` |

Với `N = 10⁵`, sort của Python (Timsort, viết bằng C) rất nhanh; cách đếm thắng không đáng kể trong thực tế.

In [20]:
rng = random.Random(1561)
big = [rng.randint(1, 10**4) for _ in range(10**5 - 10**5 % 3)]
for name, S in [("slice", SolutionSlice), ("two_ptr", SolutionTwoPointers), ("counting", SolutionCounting), ("Solution", Solution)]:
    t0 = time.perf_counter()
    r = S().maxCoins(list(big))
    print(f"{name:9s}: {r} ({time.perf_counter() - t0:.3f}s)")

slice    : 222157855 (0.020s)


two_ptr  : 222157855 (0.022s)
counting : 222157855 (0.010s)
Solution : 222157855 (0.019s)


## 12. Bộ test

In [21]:
METHODS = {
    "brute": brute_partitions,
    "sort_slice": sort_slice,
    "SolutionSlice": lambda p: SolutionSlice().maxCoins(list(p)),
    "SolutionTwoPointers": lambda p: SolutionTwoPointers().maxCoins(list(p)),
    "SolutionCounting": lambda p: SolutionCounting().maxCoins(list(p)),
    "Solution": lambda p: Solution().maxCoins(list(p)),
}
for piles, want in examples:
    for name, f in METHODS.items():
        assert f(piles) == want, name
print("ví dụ đề: OK")

ví dụ đề: OK


In [22]:
# edge case
edge = [
    ([1, 1, 1], 1),                 # 3 cọc, toàn bằng nhau
    ([1, 2, 3], 2),
    ([10000, 10000, 10000, 1, 1, 1], 10001),
    ([5, 5, 5, 5, 5, 5], 10),
    ([1, 2, 3, 4, 5, 6], 8),
    ([3, 3, 3, 3, 3, 3, 3, 3, 3], 9),
]
for piles, want in edge:
    for name, f in METHODS.items():
        assert f(piles) == want, (name, piles, f(piles))
print("edge case: OK")

edge case: OK


In [23]:
# ngẫu nhiên có seed: đối chiếu brute force (<= 12 cọc)
rng = random.Random(2019)
for _ in range(400):
    size = rng.choice([3, 6, 9, 12]) if _ % 10 else 12
    lim = rng.choice([3, 10, 10**4])
    piles = [rng.randint(1, lim) for _ in range(size)]
    ref = brute_partitions(piles)
    for name, f in METHODS.items():
        assert f(piles) == ref, (name, piles)
print("400 test ngẫu nhiên (seed 2019): mọi cách khớp brute force")

400 test ngẫu nhiên (seed 2019): mọi cách khớp brute force


In [24]:
# lớn hơn: các cách nhanh khớp nhau; hàm không phụ thuộc thứ tự input
rng = random.Random(3)
for _ in range(200):
    size = 3 * rng.randint(1, 300)
    piles = [rng.randint(1, rng.choice([5, 10**4])) for _ in range(size)]
    ref = sort_slice(piles)
    shuffled = piles[:]
    rng.shuffle(shuffled)
    for name in ["SolutionSlice", "SolutionTwoPointers", "SolutionCounting", "Solution"]:
        assert METHODS[name](piles) == ref == METHODS[name](shuffled)
print("200 mảng lớn: OK")

200 mảng lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chia bộ ba liên tiếp từ trên xuống

Sắp giảm dần rồi chia `(a₁,a₂,a₃), (a₄,a₅,a₆), ...` — Bob nhận `a₃, a₆, ...` là những cọc khá lớn, lãng phí.

In [25]:
def wrong_consecutive(piles):
    a = sorted(piles, reverse=True)
    return sum(a[i + 1] for i in range(0, len(a), 3))   # SAI: bộ ba liên tiếp

for piles, want in examples:
    print(piles, "| sai:", wrong_consecutive(piles), "| đúng:", want)

[2, 4, 1, 2, 7, 8] | sai: 9 | đúng: 9
[2, 4, 5] | sai: 4 | đúng: 4
[9, 8, 7, 6, 5, 1, 2, 3, 4] | sai: 15 | đúng: 18


Sửa: Bob nhận `n` cọc **nhỏ nhất cả mảng**, không phải cọc nhỏ nhất của từng nhóm liên tiếp.

### Lỗi 2 — Quên sắp xếp

`piles[n::2]` chỉ có nghĩa khi mảng đã sắp tăng dần.

In [26]:
def wrong_no_sort(piles):
    n = len(piles) // 3
    return sum(piles[n::2])                               # SAI: chưa sort

for piles, want in examples:
    print(piles, "| sai:", wrong_no_sort(piles), "| đúng:", want)

[2, 4, 1, 2, 7, 8] | sai: 8 | đúng: 9
[2, 4, 5] | sai: 4 | đúng: 4
[9, 8, 7, 6, 5, 1, 2, 3, 4] | sai: 10 | đúng: 18


Sửa: `piles.sort()` trước.

### Lỗi 3 — Lấy xen kẽ từ cuối nhưng không giới hạn `n` bước

`sum(a[-2::-2])` đi lùi tới tận đầu mảng ⇒ cộng cả những cọc lẽ ra của Bob.

In [27]:
def wrong_no_limit(piles):
    a = sorted(piles)
    return sum(a[-2::-2])                                 # SAI: lấy quá n cọc

for piles, want in examples:
    print(piles, "| sai:", wrong_no_limit(piles), "| đúng:", want)

[2, 4, 1, 2, 7, 8] | sai: 10 | đúng: 9
[2, 4, 5] | sai: 4 | đúng: 4
[9, 8, 7, 6, 5, 1, 2, 3, 4] | sai: 20 | đúng: 18


Sửa: `sum(a[-2::-2][:n])` hoặc `sum(a[n::2])`.

### Lỗi 4 — Lệch chỉ số bắt đầu: `a[n + 1::2]`

Bắt đầu từ `n + 1` là lấy phần của **Alice**, không phải của bạn.

In [28]:
def wrong_offset(piles):
    a = sorted(piles)
    n = len(a) // 3
    return sum(a[n + 1::2])                               # SAI: đây là phần của Alice

for piles, want in examples:
    print(piles, "| sai:", wrong_offset(piles), "| đúng:", want)

[2, 4, 1, 2, 7, 8] | sai: 12 | đúng: 9
[2, 4, 5] | sai: 5 | đúng: 4
[9, 8, 7, 6, 5, 1, 2, 3, 4] | sai: 21 | đúng: 18


Sửa: `a[n::2]`. Mẹo kiểm tra nhanh với `[2,4,5]`: `n = 1`, `a[1::2] = [4]` ✔, còn `a[2::2] = [5]` ✘.

### Lỗi 5 — Vòng hai con trỏ chạy `len(piles)` lần

Lặp theo số cọc thay vì số bước ⇒ chỉ số âm, cộng lung tung (Python cho phép chỉ số âm nên không báo lỗi!).

In [29]:
def wrong_loop_count(piles):
    a = sorted(piles)
    hi, total = len(a) - 1, 0
    for _ in range(len(a)):                               # SAI: phải là len(a) // 3
        total += a[hi - 1]
        hi -= 2
    return total

for piles, want in examples:
    print(piles, "| sai:", wrong_loop_count(piles), "| đúng:", want)

[2, 4, 1, 2, 7, 8] | sai: 20 | đúng: 9
[2, 4, 5] | sai: 11 | đúng: 4
[9, 8, 7, 6, 5, 1, 2, 3, 4] | sai: 45 | đúng: 18


Sửa: `for _ in range(len(a) // 3)` hoặc điều kiện `while lo < hi`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Bạn **chọn** nhóm, người khác lấy theo luật cố định" ⇒ không phải minimax, mà là **tối ưu phân hoạch** ⇒ thường **sắp xếp + tham lam**.
- Hỏi: "phần tử nào **chắc chắn không bao giờ** về tay mình?" (gợi ý của đề: cọc của Bob) ⇒ dồn phần tệ nhất cho nó.
- Lập luận **cận trên bằng đếm**: "phần tử lớn thứ `k` của tôi ≤ phần tử lớn thứ `f(k)` của mảng".
- Kiểm tra lát cắt bằng ví dụ nhỏ nhất (`[2,4,5]`) để không lệch chỉ số.

## 15. Biến thể của bài

1. **Mỗi bước chọn `k` cọc**, Alice lấy lớn nhất, bạn lớn thứ hai, `k − 2` người còn lại lấy phần còn lại (bài tập 5).
2. **Bạn lấy cọc lớn thứ `j`** trong bộ `k` — tổng quát: dồn `(k − j)·n` cọc nhỏ nhất cho những người sau bạn, phần còn lại chia thành khối `j` cọc, bạn lấy cọc nhỏ nhất mỗi khối.
3. **Array Partition (LC 561)**: chia cặp để tổng `min` lớn nhất — sort, lấy chỉ số chẵn.
4. **Stone Game (LC 877)**: trò chơi đối kháng thật sự (hai bên đều tối ưu) — cần DP/minimax, khác bài này.
5. **Trả về chính các bộ ba** (bài tập 4).

In [30]:
# Biến thể 3 (LC 561): chia 2n số thành n cặp, tối đa hoá tổng min mỗi cặp
def array_pair_sum(nums):
    return sum(sorted(nums)[::2])

print(array_pair_sum([1, 4, 3, 2]), array_pair_sum([6, 2, 6, 5, 1, 2]), "(LC 561: 4 và 9)")

4 9 (LC 561: 4 và 9)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Vét cạn | thử mọi phân hoạch bộ ba | siêu mũ | `O(N)` | kiểm chứng, ≤ 12 cọc |
| 2. Sort + cắt lát | `sum(a[n::2])` | `O(N log N)` | `O(N)` | **bản nộp** ngắn nhất |
| 3. Sort + hai con trỏ | mô phỏng từng bước | `O(N log N)` | `O(1)` | giải thích khi phỏng vấn |
| 4. Đếm | phát từ lớn xuống | `O(N + V)` | `O(V)` | giá trị nhỏ, cần tuyến tính |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `maxCoins([1,2,3,4,5,6])` và `maxCoins([5,5,5])`?

**Bài 2 (tính tay):** Với `piles = [3, 9, 1, 7, 4, 8, 2, 6, 5, 10, 12, 11]`, liệt kê phần Bob, Alice, bạn và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `ex3_two_pointers(piles)`.

**Bài 4 (tự viết lại):** Viết `ex4_triples(piles)` trả về danh sách `n` bộ ba (mỗi bộ là tuple 3 số lấy từ `piles`) đạt số xu tối đa cho bạn.

**Bài 5 (mini challenge):** Viết `ex5_k_group(piles, k)`: mỗi bước bạn chọn `k ≥ 2` cọc, Alice lấy cọc lớn nhất, bạn lấy lớn thứ hai, `k − 2` cọc còn lại cho người khác; `len(piles) % k == 0`. Trả về số xu tối đa của bạn.

In [31]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_two_pointers(piles):
    a = sorted(piles)
    lo, hi, total = 0, len(a) - 1, 0
    while lo < hi:
        # TODO: cộng phần của bạn trong bộ ba (a[lo], a[hi-1], a[hi])
        # TODO: dịch hi lùi 2, lo tiến 1
        pass
    return total

In [32]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_triples(piles):
    # TODO
    raise NotImplementedError

def ex5_k_group(piles, k):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Sắp xếp rồi `n = len // 3`. (2) `[1..6]`: `n = 2`, `a[2::2] = [3, 5]`. (3) `8` và `5`.

**Bài 2.** (1) Sắp: `[1..12]`, `n = 4`. (2) Bob `1,2,3,4`; còn `5..12` ghép cặp `(12,11),(10,9),(8,7),(6,5)`. (3) Bạn `11+9+7+5 = 32`.

**Bài 3.** (1) `total += a[hi - 1]`. (2) `hi -= 2`. (3) `lo += 1`.

**Bài 4.** (1) Sắp tăng dần, `n = len // 3`. (2) Bộ thứ `k` (từ 0): `(a[k], a[n + 2k], a[n + 2k + 1])`. (3) Kiểm tra: tổng phần tử giữa các bộ = `maxCoins`.

**Bài 5.** (1) Mỗi bước có `k − 2` cọc "bỏ đi" ⇒ dồn `(k − 2)·m` cọc nhỏ nhất (với `m = len/k`) cho người khác. (2) `2m` cọc lớn nhất ghép cặp (Alice, bạn) như bài gốc. (3) Đáp án `sum(a[(k−2)·m::2])`; kiểm chứng bằng vét cạn phân hoạch nhóm `k`.

## 19. Đáp án tham khảo cho bài tập

In [33]:
# Bài 1
a1 = (Solution().maxCoins([1, 2, 3, 4, 5, 6]), Solution().maxCoins([5, 5, 5]))
print("Bài 1:", a1)
assert a1 == (8, 5)

# Bài 2
p2 = [3, 9, 1, 7, 4, 8, 2, 6, 5, 10, 12, 11]
a2 = trace_two_pointers(p2)
assert a2 == 32 == brute_partitions(p2)

Bài 1: (8, 5)
sắp xếp: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  vòng 1: lo=0, hi=11 -> bộ ba (Bob 1, bạn 11, Alice 12), tổng = 11
  vòng 2: lo=1, hi=9 -> bộ ba (Bob 2, bạn 9, Alice 10), tổng = 20
  vòng 3: lo=2, hi=7 -> bộ ba (Bob 3, bạn 7, Alice 8), tổng = 27
  vòng 4: lo=3, hi=5 -> bộ ba (Bob 4, bạn 5, Alice 6), tổng = 32
  dừng: lo=4, hi=3, đáp án 32


In [34]:
# Bài 3
def ex3_two_pointers_sol(piles):
    a = sorted(piles)
    lo, hi, total = 0, len(a) - 1, 0
    while lo < hi:
        total += a[hi - 1]
        hi -= 2
        lo += 1
    return total

rng3 = random.Random(3)
for _ in range(300):
    piles = [rng3.randint(1, 30) for _ in range(3 * rng3.randint(1, 4))]
    assert ex3_two_pointers_sol(piles) == brute_partitions(piles)
print("Bài 3: OK")

Bài 3: OK


In [35]:
# Bài 4
def ex4_triples_sol(piles):
    a = sorted(piles)
    n = len(a) // 3
    return [(a[k], a[n + 2 * k], a[n + 2 * k + 1]) for k in range(n)]

from collections import Counter
rng4 = random.Random(4)
for _ in range(500):
    piles = [rng4.randint(1, 50) for _ in range(3 * rng4.randint(1, 6))]
    tr = ex4_triples_sol(piles)
    assert Counter(x for t in tr for x in t) == Counter(piles)          # dùng đúng mọi cọc
    assert sum(sorted(t)[1] for t in tr) == Solution().maxCoins(list(piles))
print("Bài 4: OK, ví dụ 1:", ex4_triples_sol([2, 4, 1, 2, 7, 8]))

Bài 4: OK, ví dụ 1: [(1, 2, 4), (2, 7, 8)]


In [36]:
# Bài 5
def ex5_k_group_sol(piles, k):
    a = sorted(piles)
    m = len(a) // k
    return sum(a[(k - 2) * m::2])

def ex5_brute(piles, k):
    def solve(rest):
        if not rest:
            return 0
        x, others = rest[0], rest[1:]
        best = 0
        for idx in combinations(range(len(others)), k - 1):
            grp = sorted([x] + [others[i] for i in idx])
            remain = [others[i] for i in range(len(others)) if i not in idx]
            best = max(best, grp[-2] + solve(remain))
        return best
    return solve(list(piles))

assert all(ex5_k_group_sol(p, 3) == w for p, w in examples)
rng5 = random.Random(5)
for _ in range(300):
    k = rng5.randint(2, 4)
    m = rng5.randint(1, 8 // k + (1 if k == 2 else 0))
    piles = [rng5.randint(1, 20) for _ in range(k * m)]
    assert ex5_k_group_sol(piles, k) == ex5_brute(piles, k), (piles, k)
print("Bài 5: OK, ví dụ k=4:", ex5_k_group_sol([1, 2, 3, 4, 5, 6, 7, 8], 4))

Bài 5: OK, ví dụ k=4: 12


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def maxCoins(self, piles):
        piles.sort()
        return sum(piles[len(piles) // 3 :: 2])
```

**Template "tôi lấy phần thứ j trong nhóm do tôi chọn":** sắp xếp; dồn những phần tử chắc chắn không về tay mình (nhỏ nhất) cho người sau; phần còn lại chia khối từ trên xuống; chứng minh bằng cận trên đếm.

**Checklist trước khi submit:**
- [ ] Đã sắp xếp tăng dần.
- [ ] Lát cắt bắt đầu ở `n = len // 3` (không phải `n + 1`), bước `2`.
- [ ] Nếu dùng vòng lặp: đúng `n` bước.
- [ ] Thử nhanh: `[2,4,5] → 4`, `[2,4,1,2,7,8] → 9`, `[9,8,7,6,5,1,2,3,4] → 18`.